# Guitar Pedal Mel Spectrogram Training

This notebook contains the PedalResNet model and training code.

In [2]:
# src/Classification/model.py
import torch
import torch.nn as nn
import torchvision.models as models
from typing import Union
from pathlib import Path


class PedalResNet(nn.Module):
    """
    ResNet34 adapted for 1-channel Mel spectrogram input and 2-value regression output:
    [drive, tone].

    NOTE: This architecture is matched to the one used in melTrain.py.
    """

    def __init__(self, output_size=2, use_pretrained=False):
        super().__init__()

        if use_pretrained:
            base = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        else:
            base = models.resnet34(weights=None)

        base.conv1 = nn.Conv2d(
            1, 64, kernel_size=7, stride=2, padding=3, bias=False
        )

        base.fc = nn.Linear(base.fc.in_features, output_size)
        self.resnet = base

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.resnet(x)

    def load_weights(self, path: Union[str, Path], map_location: str = "cpu"):
        state = torch.load(path, map_location=map_location)

        if isinstance(state, dict) and "state_dict" in state:
            state = state["state_dict"]

        filtered_state = {}
        for k, v in state.items():
            if k == "conv1.weight" and v.shape != self.resnet.conv1.weight.shape:
                continue
            filtered_state[k] = v

        self.resnet.load_state_dict(filtered_state, strict=False)
        self.eval()
        return self


In [28]:
import warnings
import sys
from pathlib import Path

project_root = Path.cwd()
sys.path.append(str(project_root / "src"))

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import models
from tqdm import tqdm
from Classification.melDataLoader import GuitarPedalDataset

from select_path import load_config

warnings.filterwarnings("ignore", category=UserWarning)

root = load_config()
dist = root / "distorted"


ModuleNotFoundError: No module named 'melSpec'

In [16]:
def train_model(
    data_dir,
    num_epochs: int = 100,
    batch_size: int = 8,
    lr: float = 1e-4,
    model_name: str = "resnet34",
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dataset = GuitarPedalDataset(data_dir)
    total_size = len(dataset)

    train_size = int(0.8 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    generator = torch.Generator().manual_seed(42)
    train_set, val_set, test_set = random_split(
        dataset, [train_size, val_size, test_size], generator=generator
    )

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size)
    test_loader = DataLoader(test_set, batch_size=batch_size)

    model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model = model.to(device)

    criterion = nn.SmoothL1Loss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(num_epochs):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            if x.ndim == 3:
                x = x.unsqueeze(1)
            optimizer.zero_grad()
            loss = criterion(model(x), y / 10.0)
            loss.backward()
            optimizer.step()

    model.eval()


In [ ]:
if __name__ == "__main__":
    train_model(data_dir=dist, model_name="resnet34")
